# $\mathrm{K}_{\mathrm{V}}2.1$ F412L S6-neighborhood distance analysis

Default ensemble subset: **allOk3 + structural/interface QC** (`all_ok_3_structural_interface_qc`; corrected 3 Å convergence, G377 tetramer-integrity filtering, and a 27 Å pore–VSD interface cutoff).
The same residue pairs are used for four complementary comparisons: WT versus F412L within each protocol, and vanilla versus masked within each construct. The 8SD3 WT and 8SDA L403A structures are overlaid as experimental state references because L403A and F412L were proposed to perturb related S6 rearrangements. These references test whether F412L samples an L403A-like direction; 8SDA is not a mutation-matched F412L structure.


## Comparison design

The analysis separates two effects that should not be conflated. WT and F412L
are compared within vanilla and masked protocols to estimate the mutation
effect, while vanilla and masked ensembles are compared within WT and F412L to
estimate the masking effect. The same ranked Cα-distance panel is retained in
all four comparisons, and 8SD3 WT and 8SDA L403A values provide fixed
experimental state references.

### WT–F412L hydrophobic-nexus comparison

The image below loads the current publication-resolution export, including the WT and F412L vanilla and masked ensembles and the Kv2.1 experimental references.

![WT versus F412L hydrophobic-nexus geometry](dataRMSD/analysis/nexus/figures/f412l_nexus_geometry_vs_experiment.png)


### F412L-only hydrophobic-nexus comparison

This focused version removes the WT prediction distributions. Each violin places F412L vanilla on the left and F412L masked on the right. The 8SD3 WT and 8SDA L403A measurements remain as structural references rather than mutation-matched observations.

![F412L-only hydrophobic-nexus geometry](dataRMSD/analysis/nexus/figures/f412l_only_nexus_geometry_vs_experiment.png)


In [ ]:
from pathlib import Path
import sys
import pandas as pd
repo_root = Path.cwd() if (Path.cwd() / 'shared').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root)) if str(repo_root) not in sys.path else None
from shared.mutation_site_analysis import nearby_distance_columns, rank_nearby_shifts, top_aliases
from shared.data_access import resolve
import importlib, shared.dataset_selection as shared_dataset_selection
importlib.reload(shared_dataset_selection)
from shared.dataset_selection import distance_csv_options, load_selected_distance_csv, kv21_active_masked_distance_path, kv21_active_rmsd_path
from shared.plotting import KV21_PALETTE, apply_kv21_style, experimental_reference_style, plot_split_ensemble_with_experimentals, plot_protocol_split_with_experimentals
apply_kv21_style()


In [ ]:
from shared.presentation_export import install_notebook_figure_export
WRITING_FIGURE_DIR = repo_root / 'docs' / 'figures' / 'notebook_exports' / 'kv21'
install_notebook_figure_export(WRITING_FIGURE_DIR, 'Kv21_F412L_mutationSite_analysis')


In [ ]:
DATASET_SELECTION = 'all_ok_3'  # Corrected 3 Å convergence cohort; corrected structural/interface-QC joins are not staged yet.
data = repo_root / 'kv21' / 'dataDistances'
DATASET_PATHS = {
    'WT vanilla': distance_csv_options(repo_root, data / '26-02-11_Kv2.1_wt_vanillaAF2_distances_all.csv', 'Kv21', 'WT', 'vanilla'),
    'WT masked': distance_csv_options(repo_root, kv21_active_masked_distance_path(repo_root, 'WT'), 'Kv21', 'WT', 'masked'),
    'F412L vanilla': distance_csv_options(repo_root, data / '26-02-11_Kv2.1_f412l_vanillaAF2_distances_all.csv', 'Kv21', 'F412L', 'vanilla'),
    'F412L masked': distance_csv_options(repo_root, kv21_active_masked_distance_path(repo_root, 'F412L'), 'Kv21', 'F412L', 'masked'),
}
DATASET_PATHS
wt_van = load_selected_distance_csv('WT vanilla', DATASET_PATHS['WT vanilla'], DATASET_SELECTION)
wt_mask = load_selected_distance_csv('WT masked', DATASET_PATHS['WT masked'], DATASET_SELECTION)
mut_van = load_selected_distance_csv('F412L vanilla', DATASET_PATHS['F412L vanilla'], DATASET_SELECTION)
mut_mask = load_selected_distance_csv('F412L masked', DATASET_PATHS['F412L masked'], DATASET_SELECTION)


## S6-neighborhood distributions

Candidate coordinates are ranked by the largest absolute WT-to-F412L median
shift observed in either protocol. Split violins show the complete retained
ensemble rather than only summary statistics. The experimental 8SDA points test
whether F412L samples an L403A-like S6 direction; they are not a
mutation-matched F412L validation structure.

In [ ]:
import importlib
from shared.structure_distances import experimental_distances_for_aliases as experimental_for_aliases

# Select one compact set of F412-adjacent distances for every panel. Ranking by the
# largest shift seen in either protocol avoids changing the x axis between comparisons.
vanilla_shifts = rank_nearby_shifts(wt_van, mut_van, nearby_distance_columns(wt_van, 412))
masked_shifts = rank_nearby_shifts(wt_mask, mut_mask, nearby_distance_columns(wt_mask, 412))
combined_shifts = pd.concat([vanilla_shifts, masked_shifts], ignore_index=True)
combined_shifts = combined_shifts[combined_shifts['distance'].str.startswith('CA_CA_')]
combined_shifts = combined_shifts.sort_values('abs_shift_A', ascending=False).drop_duplicates('distance')
aliases = top_aliases(combined_shifts, n=12, residue_number_offset=-2)
display(vanilla_shifts.head(12).assign(protocol='vanilla'))
display(masked_shifts.head(12).assign(protocol='masked'))

# Calculate the matching Cα distances directly from the experimental coordinates.
# Visible aliases already use the experimental/paper numbering (model numbering − 2).
import math, re


exp_wt, missing_wt = experimental_for_aliases(aliases, repo_root/'kv21'/'experimental'/'8SD3.pdb')
exp_l403a, missing_l403a = experimental_for_aliases(aliases, repo_root/'kv21'/'experimental'/'8SDA.pdb')
print(f'Experimental matches: 8SD3={len(exp_wt)}/{len(aliases)}, 8SDA={len(exp_l403a)}/{len(aliases)}')
if missing_wt or missing_l403a:
    print('Unresolved aliases:', sorted(set(missing_wt + missing_l403a)))
experimental_distances = [exp_wt, exp_l403a]
experimental_labels = ['Experimental | 8SD3 WT', 'Experimental | 8SDA L403A state reference']
experimental_colors = [experimental_reference_style(reference)['color'] for reference in ('8SD3', '8SDA')]
region = 'F412-adjacent S6 neighborhood'

# Mutation effect within each prediction protocol.
plot_split_ensemble_with_experimentals(wt_van, mut_van, aliases, 'F412L', 'vanilla', region, experimental_distances, experimental_labels, experimental_colors)
plot_split_ensemble_with_experimentals(wt_mask, mut_mask, aliases, 'F412L', 'masked', region, experimental_distances, experimental_labels, experimental_colors)

# Masking effect within each sequence background.
plot_protocol_split_with_experimentals(wt_van, wt_mask, aliases, 'Kv2.1 | WT | vanilla vs masked | F412-adjacent S6 neighborhood | experimental state references', [KV21_PALETTE['WT_VAN'], KV21_PALETTE['WT_HM']], experimental_distances, experimental_labels, experimental_colors)
plot_protocol_split_with_experimentals(mut_van, mut_mask, aliases, 'Kv2.1 | F412L | vanilla vs masked | F412-adjacent S6 neighborhood | experimental state references', [KV21_PALETTE['F412L_VAN'], KV21_PALETTE['F412L_HM']], experimental_distances, experimental_labels, experimental_colors)

## Contacts altered by F412L (model/CSV F414L)
import shared.plotting as shared_plotting
importlib.reload(shared_plotting)
import shared.mutation_contact_analysis as mutation_contact_analysis
importlib.reload(mutation_contact_analysis)
from shared.mutation_contact_analysis import plot_mutation_contacts, plot_selected_contact_distributions
f412l_contact_table, _ = plot_mutation_contacts(
    {'vanilla': (wt_van, mut_van), 'masked': (wt_mask, mut_mask)},
    'PHE414A', 'LEU414A', 'F412L',
    {'vanilla': KV21_PALETTE['F412L_VAN'], 'masked': KV21_PALETTE['F412L_HM']}, channel='Kv2.1',
)
display(f412l_contact_table.head(15).round(3))
f412l_top_partners = list(dict.fromkeys(f412l_contact_table['Partner']))[:3]
f412l_contact_distributions, f412l_contact_figure = plot_selected_contact_distributions(
    {'vanilla': (wt_van, mut_van), 'masked': (wt_mask, mut_mask)},
    'PHE414A', 'LEU414A', 'F412L', f412l_top_partners,
    {'vanilla': (KV21_PALETTE['WT_VAN'], KV21_PALETTE['F412L_VAN']), 'masked': (KV21_PALETTE['WT_HM'], KV21_PALETTE['F412L_HM'])}, channel='Kv2.1',
    show_overlap_region=False,
)
display(f412l_contact_figure)
contact_figure_dir = repo_root / 'kv21' / 'dataDistances' / 'analysis' / 'figures'
contact_figure_dir.mkdir(parents=True, exist_ok=True)
f412l_contact_figure.savefig(
    contact_figure_dir / 'f412l_mutation_site_shortest_distances.png',
    dpi=400, bbox_inches='tight',
)


## Hydrophobic coupling nexus: experimental benchmark

F412 sits at the intracellular end of S6, where it couples to L316 in the S4–S5 linker and to L329 and L403 in the neighboring pore-forming subunit. This compact hydrophobic network links voltage-sensor movement to the S6 gate. The F412L mutant was functionally severe, but the corresponding cryo-EM reconstruction could not be resolved. The L403A structure (8SDA) therefore provides the closest experimental view of how weakening this nexus can promote inactivation.

The experimental comparison is asymmetric. In 8SD3, all three contacts are close to 3.8–3.9 Å in every subunit. In 8SDA, the same-chain F412–L316 contact remains compact, whereas the nearest intersubunit F412–L329 and F412–L403 contacts spread over approximately 3.5–7.3 Å. Thus, the structural signature is not uniform expansion of the entire nexus. It is retention of the S4–S5-linker contact together with asymmetric loss or redistribution of the neighboring S5/S6 contacts.

Two complementary tests are used below. First, for each model and region,

\[
\Delta_{\mathrm{reference}} = \mathrm{RMSD}_{8\mathrm{SDA}}-\mathrm{RMSD}_{8\mathrm{SD3}}.
\]

Negative values identify models locally closer to the L403A structure; positive values identify models closer to WT. Second, shortest heavy-atom distances test whether the three physical contacts are retained. In the nexus-distance figure, the left violin half represents WT F412 and the right half represents F412L L412, with vanilla and masked ensembles in separate panels. Cross-chain labels are not assumed to correspond between 8SD3, 8SDA, and AlphaFold models; the closest available partner in each tetramer is used. This avoids converting a cyclic chain-label change into a false contact loss.

The representative-model score is the mean Cα RMSD to 8SDA across the hydrophobic nexus, L403 region, and F412 region after stable-core alignment. It is useful for choosing structures to inspect, but it is not an all-atom side-chain RMSD and should not be interpreted as proof that every rotamer is correct.

In [ ]:
nexus_relpath = str(kv21_active_rmsd_path(repo_root, 'F412L'))
nexus_source = resolve(nexus_relpath) if nexus_relpath.strip() and nexus_relpath.lower() != 'nan' else None
if nexus_source is not None and nexus_source.exists():
    import importlib
    import shared.rmsd_analysis as shared_rmsd_analysis
    importlib.reload(shared_rmsd_analysis)
    import shared.kv21_nexus_analysis as kv21_nexus_analysis
    importlib.reload(kv21_nexus_analysis)
    write_nexus_outputs = kv21_nexus_analysis.write_nexus_outputs
    
    nexus = write_nexus_outputs(repo_root)
    display(nexus["preference"].round(3))
    display(nexus["contacts"].round(3))
    display(nexus["experimental"].round(3))
    display(nexus["representatives"].groupby(["sequence_condition", "protocol"]).head(1).round(3))
    display(nexus["preference_figure"])
    display(nexus["geometry_figure"])
    display(nexus["mutant_geometry_figure"])
    display(nexus["contact_figure"])
else:
    print('Hydrophobic-nexus recomputation skipped: public RMSD source is absent; checked-in nexus figures remain the archived source.')
